# ApexPlanet Task 2 – SQL for Data Extraction
Run all cells from top to bottom. Upload Task 1's cleaned Superstore CSV when prompted.

## 1. Install and import

In [1]:
%pip -q install sqlalchemy pandas
import os, re, pandas as pd
from sqlalchemy import create_engine, text
from google.colab import files
print("Setup complete")

Setup complete


## 2. Upload Task 1 cleaned CSV

In [20]:
csv_path = "/content/superstore_cleaned.csv"
if not os.path.exists(csv_path):
    print("Choose Task 1's superstore_cleaned.csv")
    uploaded = files.upload()
    csv_path = next(iter(uploaded))
df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (10296, 25)
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Order Year', 'Order Month', 'Shipping Days', 'Profit Margin']


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order Year,Order Month,Shipping Days,Profit Margin
0,1.0,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136,2017.0,2017-11,3.0,0.1600
1,2.0,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820,2017.0,2017-11,3.0,0.3000
2,3.0,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714,2017.0,2017-06,4.0,0.4700
3,4.0,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310,2016.0,2016-10,7.0,-0.4000
4,5.0,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164,2016.0,2016-10,7.0,0.1125


## 3. Standardize columns and prepare data

In [8]:
def clean_name(x):
    return re.sub(r"_+", "_", re.sub(r"[^a-z0-9]+", "_", str(x).strip().lower())).strip("_")
df.columns = [clean_name(c) for c in df.columns]
# Recognize common alternate column spellings
aliases = {
 "orderdate":"order_date", "orderid":"order_id", "customerid":"customer_id",
 "customer":"customer_name", "customername":"customer_name",
 "product":"product_name", "productname":"product_name",
 "subcategory":"sub_category", "shipmode":"ship_mode"
}
df = df.rename(columns={c: aliases[c] for c in df.columns if c in aliases})
required = ["order_date","sales","profit","category","product_name"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Found columns: {df.columns.tolist()}. Please upload Task 1 cleaned Superstore CSV.")
df["sales"] = pd.to_numeric(df["sales"], errors="coerce")
df["profit"] = pd.to_numeric(df["profit"], errors="coerce")
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df = df.dropna(subset=["sales","profit","order_date"])
df["order_month"] = df["order_date"].dt.strftime("%Y-%m")
for c in ["quantity","discount"]:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
print("Prepared data:", df.shape)
print("Standardized columns:", df.columns.tolist())
display(df.head())

Prepared data: (9994, 25)
Standardized columns: ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', 'order_year', 'order_month', 'shipping_days', 'profit_margin']


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,sub_category,product_name,sales,quantity,discount,profit,order_year,order_month,shipping_days,profit_margin
0,1.0,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136,2017.0,2017-11,3.0,0.1600
1,2.0,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820,2017.0,2017-11,3.0,0.3000
2,3.0,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714,2017.0,2017-06,4.0,0.4700
3,4.0,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310,2016.0,2016-10,7.0,-0.4000
4,5.0,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164,2016.0,2016-10,7.0,0.1125


## 4. Create SQLite database using SQLAlchemy

In [9]:
engine = create_engine("sqlite:////content/superstore_task2.db")
df.to_sql("superstore", engine, if_exists="replace", index=False)
with engine.connect() as con:
    print("Rows loaded:", con.execute(text("SELECT COUNT(*) FROM superstore")).scalar())

Rows loaded: 9994


## 5. SELECT, WHERE, ORDER BY, LIMIT

In [10]:
q = '''
SELECT product_name, sales, profit
FROM superstore
WHERE sales > 500
ORDER BY sales DESC
LIMIT 10
'''
with engine.connect() as con: display(pd.read_sql_query(text(q), con))

,product_name,sales,profit
0,Cisco TelePresence System EX90 Videoconferenci...,22638.480,-1811.0784
1,Canon imageCLASS 2200 Advanced Copier,17499.950,8399.9760
2,Canon imageCLASS 2200 Advanced Copier,13999.960,6719.9808
3,Canon imageCLASS 2200 Advanced Copier,11199.968,3919.9888
4,Canon imageCLASS 2200 Advanced Copier,10499.970,5039.9856
5,GBC Ibimaster 500 Manual ProClick Binding System,9892.740,4946.3700
6,Ibico EPK-21 Electric Binding System,9449.950,4630.4755
7,"3D Systems Cube Printer, 2nd Generation, Magenta",9099.930,2365.9818
8,HP Designjet T520 Inkjet Large Format Printer ...,8749.950,2799.9840
9,Canon imageCLASS 2200 Advanced Copier,8399.976,1119.9968


## 6. GROUP BY and HAVING

In [11]:
q = '''
SELECT category, ROUND(SUM(sales),2) AS total_sales,
       ROUND(SUM(profit),2) AS total_profit
FROM superstore
GROUP BY category
HAVING SUM(sales) > 0
ORDER BY total_sales DESC
'''
with engine.connect() as con: display(pd.read_sql_query(text(q), con))

,category,total_sales,total_profit
0,Technology,836154.03,145454.95
1,Furniture,741999.80,18451.27
2,Office Supplies,719047.03,122490.80


## 7. JOIN – customer spending

In [12]:
if {"customer_id","customer_name"}.issubset(df.columns):
    q = '''
    SELECT customer_id, customer_name, SUM(sales) AS total_spend
    FROM superstore GROUP BY customer_id, customer_name
    ORDER BY total_spend DESC LIMIT 10
    '''
    with engine.connect() as con: display(pd.read_sql_query(text(q), con))
else:
    print("JOIN example skipped: customer_id/customer_name columns are unavailable.")

,customer_id,customer_name,total_spend
0,SM-20320,Sean Miller,25043.050
1,TC-20980,Tamara Chand,19052.218
2,RB-19360,Raymond Buch,15117.339
3,TA-21385,Tom Ashbrook,14595.620
4,AB-10105,Adrian Barton,14473.571
5,KL-16645,Ken Lonsdale,14175.229
6,SC-20095,Sanjit Chand,14142.334
7,HL-15040,Hunter Lopez,12873.298
8,SE-20110,Sanjit Engle,12209.438
9,CC-12370,Christopher Conant,12129.072


## 8. Subquery

In [13]:
q = '''
SELECT product_name, SUM(sales) AS total_sales
FROM superstore GROUP BY product_name
HAVING SUM(sales) > (
  SELECT AVG(product_total) FROM (
    SELECT SUM(sales) AS product_total
    FROM superstore GROUP BY product_name
  )
)
ORDER BY total_sales DESC LIMIT 10
'''
with engine.connect() as con: display(pd.read_sql_query(text(q), con))

,product_name,total_sales
0,Canon imageCLASS 2200 Advanced Copier,61599.824
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.384
2,Cisco TelePresence System EX90 Videoconferenci...,22638.480
3,HON 5400 Series Task Chairs for Big and Tall,21870.576
4,GBC DocuBind TL300 Electric Binding System,19823.479
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.500
6,Hewlett Packard LaserJet 3310 Copier,18839.686
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.895
8,GBC DocuBind P400 Electric Binding System,17965.068
9,High Speed Automatic Electric Letter Opener,17030.312


## 9. CTE – monthly sales

In [14]:
q = '''
WITH monthly AS (
 SELECT order_month, SUM(sales) AS total_sales
 FROM superstore GROUP BY order_month
)
SELECT order_month, ROUND(total_sales,2) AS total_sales
FROM monthly ORDER BY order_month
'''
with engine.connect() as con:
    monthly_sales = pd.read_sql_query(text(q), con)
display(monthly_sales)

,order_month,total_sales
0,2015-01,14236.90
1,2015-02,4519.89
2,2015-03,55691.01
3,2015-04,28295.35
4,2015-05,23648.29
5,2015-06,34595.13
6,2015-07,33946.39
7,2015-08,27909.47
8,2015-09,81777.35
9,2015-10,31453.39


## 10. Window functions – ROW_NUMBER and RANK

In [15]:
q = '''
WITH products AS (
 SELECT category, product_name, SUM(sales) AS total_sales
 FROM superstore GROUP BY category, product_name
)
SELECT category, product_name, ROUND(total_sales,2) AS total_sales,
 ROW_NUMBER() OVER (PARTITION BY category ORDER BY total_sales DESC) AS row_num,
 RANK() OVER (PARTITION BY category ORDER BY total_sales DESC) AS sales_rank
FROM products ORDER BY category, sales_rank
'''
with engine.connect() as con: display(pd.read_sql_query(text(q), con).head(20))

,category,product_name,total_sales,row_num,sales_rank
0,Furniture,HON 5400 Series Task Chairs for Big and Tall,21870.58,1,1
1,Furniture,"Riverside Palais Royal Lawyers Bookcase, Royal...",15610.97,2,2
2,Furniture,Bretford Rectangular Conference Table Tops,12995.29,3,3
3,Furniture,Global Troy Executive Leather Low-Back Tilter,12975.38,4,4
4,Furniture,SAFCO Arco Folding Chair,11572.78,5,5
5,Furniture,DMI Eclipse Executive Suite Bookcases,11046.61,6,6
6,Furniture,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",10637.53,7,7
7,Furniture,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,8,8
8,Furniture,Bush Advantage Collection Racetrack Conference...,9544.73,9,9
9,Furniture,GuestStacker Chair with Chrome Finish Legs,9070.94,10,10


## 11. Window functions – LAG and LEAD

In [16]:
q = '''
WITH monthly AS (
 SELECT order_month, SUM(sales) AS total_sales
 FROM superstore GROUP BY order_month
)
SELECT order_month, ROUND(total_sales,2) AS total_sales,
 ROUND(LAG(total_sales) OVER (ORDER BY order_month),2) AS previous_month,
 ROUND(LEAD(total_sales) OVER (ORDER BY order_month),2) AS next_month
FROM monthly ORDER BY order_month
'''
with engine.connect() as con: display(pd.read_sql_query(text(q), con))

,order_month,total_sales,previous_month,next_month
0,2015-01,14236.90,NaN,4519.89
1,2015-02,4519.89,14236.90,55691.01
2,2015-03,55691.01,4519.89,28295.35
3,2015-04,28295.35,55691.01,23648.29
4,2015-05,23648.29,28295.35,34595.13
5,2015-06,34595.13,23648.29,33946.39
6,2015-07,33946.39,34595.13,27909.47
7,2015-08,27909.47,33946.39,81777.35
8,2015-09,81777.35,27909.47,31453.39
9,2015-10,31453.39,81777.35,78628.72


## 12. SQL View

In [17]:
with engine.begin() as con:
    con.execute(text("DROP VIEW IF EXISTS category_performance"))
    con.execute(text('''
    CREATE VIEW category_performance AS
    SELECT category, SUM(sales) AS total_sales, SUM(profit) AS total_profit
    FROM superstore GROUP BY category
    '''))
with engine.connect() as con:
    display(pd.read_sql_query(text("SELECT * FROM category_performance ORDER BY total_sales DESC"), con))

,category,total_sales,total_profit
0,Technology,836154.0330,145454.9481
1,Furniture,741999.7953,18451.2728
2,Office Supplies,719047.0320,122490.8008


## 13. Ten business questions

In [18]:
business_queries = {
"Top 5 products by sales": "SELECT product_name, SUM(sales) AS sales FROM superstore GROUP BY product_name ORDER BY sales DESC LIMIT 5",
"Monthly sales trend": "SELECT order_month, SUM(sales) AS sales FROM superstore GROUP BY order_month ORDER BY order_month",
"Customer segmentation by spend": "SELECT segment, SUM(sales) AS spend FROM superstore GROUP BY segment ORDER BY spend DESC" if "segment" in df.columns else None,
"Sales by region": "SELECT region, SUM(sales) AS sales FROM superstore GROUP BY region ORDER BY sales DESC" if "region" in df.columns else None,
"Profit by category": "SELECT category, SUM(profit) AS profit FROM superstore GROUP BY category ORDER BY profit DESC",
"Top 10 customers": "SELECT customer_name, SUM(sales) AS sales FROM superstore GROUP BY customer_name ORDER BY sales DESC LIMIT 10" if "customer_name" in df.columns else None,
"Quantity by category": "SELECT category, SUM(quantity) AS quantity FROM superstore GROUP BY category ORDER BY quantity DESC" if "quantity" in df.columns else None,
"Average discount by category": "SELECT category, AVG(discount) AS avg_discount FROM superstore GROUP BY category" if "discount" in df.columns else None,
"Profit by sub-category": "SELECT sub_category, SUM(profit) AS profit FROM superstore GROUP BY sub_category ORDER BY profit DESC" if "sub_category" in df.columns else None,
"Orders by ship mode": "SELECT ship_mode, COUNT(DISTINCT order_id) AS orders FROM superstore GROUP BY ship_mode" if {"ship_mode","order_id"}.issubset(df.columns) else None
}
for title, q in business_queries.items():
    print("\n" + title)
    if q is None:
        print("Skipped because required dataset column is unavailable.")
    else:
        with engine.connect() as con: display(pd.read_sql_query(text(q), con))


Top 5 products by sales


,product_name,sales
0,Canon imageCLASS 2200 Advanced Copier,61599.824
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.384
2,Cisco TelePresence System EX90 Videoconferenci...,22638.480
3,HON 5400 Series Task Chairs for Big and Tall,21870.576
4,GBC DocuBind TL300 Electric Binding System,19823.479



Monthly sales trend


,order_month,sales
0,2015-01,14236.8950
1,2015-02,4519.8920
2,2015-03,55691.0090
3,2015-04,28295.3450
4,2015-05,23648.2870
5,2015-06,34595.1276
6,2015-07,33946.3930
7,2015-08,27909.4685
8,2015-09,81777.3508
9,2015-10,31453.3930



Customer segmentation by spend


,segment,spend
0,Consumer,1.161401e+06
1,Corporate,7.061464e+05
2,Home Office,4.296531e+05


... (10 outputs left out)


Profit by sub-category


,sub_category,profit
0,Copiers,55617.8249
1,Phones,44515.7306
2,Accessories,41936.6357
3,Paper,34053.5693
4,Binders,30221.7633
5,Chairs,26590.1663
6,Storage,21278.8264
7,Appliances,18138.0054
8,Furnishings,13059.1436
9,Envelopes,6964.1767



Orders by ship mode


,ship_mode,orders
0,First Class,787
1,Same Day,264
2,Second Class,964
3,Standard Class,2994


## 14. Export outputs

In [19]:
q = business_queries["Top 5 products by sales"]
with engine.connect() as con:
    top5 = pd.read_sql_query(text(q), con)
top5.to_csv("/content/top_5_products.csv", index=False)
files.download("/content/top_5_products.csv")
files.download("/content/superstore_task2.db")
print("Task 2 queries completed successfully.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Task 2 queries completed successfully.
